In [49]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

EPOCHS = 50
BATCH = 16
IMG_SIZE = 224
NUM_CLASSES = 38


In [50]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", device)

DEVICE: cuda


In [51]:
test_val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE,IMG_SIZE)),
        transforms.ToTensor(),
])

test_ds  = datasets.ImageFolder(fr"michal/nowe/100/test", transform=test_val_transform)

test_loader  = DataLoader(test_ds, batch_size=BATCH)


In [52]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
import time
import numpy as np 
def evaluate_model(model, dataloader, device):

    model.eval()

    y_true = []
    y_pred = []
    times = []
    with torch.no_grad():

        for x, y in dataloader:
            
            x = x.to(device)
            y = y.to(device)
            start = time.time()
            outputs = model(x)
            end = time.time()
            preds = torch.argmax(outputs, dim=1)
            times.append(end - start)
            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    micro_f1 = f1_score(y_true, y_pred, average="micro")
    recall = recall_score(y_true, y_pred, average="micro")
    precision = precision_score(y_true, y_pred, average="micro")
    mean_time = np.mean(times)
    num_params = sum(p.numel() for p in model.parameters())

    return accuracy, macro_f1, micro_f1, recall, precision, mean_time ,num_params


In [53]:
import os

MODEL_DIR = fr"output"
MODEL_PREFIX = "resnet101"

model_files = [
    f for f in os.listdir(MODEL_DIR)
    if f.endswith(".pth") and f.startswith(MODEL_PREFIX)
]
print(model_files)

['resnet101_100.pth', 'resnet101_25.pth', 'resnet101_25_noise.pth', 'resnet101_50.pth', 'resnet101_50_noise.pth', 'resnet101_75_noise.pth']


In [54]:
import pandas as pd

import os

MODEL_DIR = fr"output"
MODEL_PREFIX = "resnet101"

model_files = [
    f for f in os.listdir(MODEL_DIR)
    if f.endswith(".pth") and f.startswith(MODEL_PREFIX)
]
print(model_files)

results = []
for file_name in model_files:

    path = os.path.join(MODEL_DIR, file_name)


    model = models.resnet101(weights=None)
    model.fc = nn.Linear(
            model.fc.in_features,
            NUM_CLASSES
    )

    state_dict = torch.load(path, map_location=device)

    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(state_dict)

    model.to(device)
    model.eval()

    accuracy, macro_f1, micro_f1, recall, precision, mean_time, param = evaluate_model(
        model,
        test_loader,
        device
    )

    results.append({
        "file_name": file_name,
        "accuracy": round(accuracy, 4),
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "recall" : round(recall, 4),
        "presicion" : round(precision, 4),
        "time" : round(mean_time,4),
        "parameters" : param
    })

df = pd.DataFrame(results)

df = df.sort_values(by="file_name", ascending=False)

print("\nRESULTS:\n")

print(df.to_string(index=False))

['resnet101_100.pth', 'resnet101_25.pth', 'resnet101_25_noise.pth', 'resnet101_50.pth', 'resnet101_50_noise.pth', 'resnet101_75_noise.pth']


C:\Users\kubac\AppData\Local\Temp\ipykernel_33380\3842839852.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)
C:\Users


RESULTS:

             file_name  accuracy  macro_f1  micro_f1  recall  presicion   time  parameters
resnet101_75_noise.pth    0.9965    0.9960    0.9965  0.9965     0.9965 0.0242    42578022
resnet101_50_noise.pth    0.9976    0.9964    0.9976  0.9976     0.9976 0.0239    42578022
      resnet101_50.pth    0.9914    0.9869    0.9914  0.9914     0.9914 0.0244    42578022
resnet101_25_noise.pth    0.9949    0.9927    0.9949  0.9949     0.9949 0.0240    42578022
      resnet101_25.pth    0.9903    0.9860    0.9903  0.9903     0.9903 0.0230    42578022
     resnet101_100.pth    0.9962    0.9961    0.9962  0.9962     0.9962 0.0224    42578022


In [55]:
import pandas as pd

import os

MODEL_DIR = fr"output"
MODEL_PREFIX = "resnet50"

model_files = [
    f for f in os.listdir(MODEL_DIR)
    if f.endswith(".pth") and f.startswith(MODEL_PREFIX)
]
print(model_files)

results = []
for file_name in model_files:

    path = os.path.join(MODEL_DIR, file_name)


    model = models.resnet50(weights=None)
    model.fc = nn.Linear(
            model.fc.in_features,
            NUM_CLASSES
    )

    state_dict = torch.load(path, map_location=device)

    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(state_dict)

    model.to(device)
    model.eval()

    accuracy, macro_f1, micro_f1, recall, precision, mean_time, param = evaluate_model(
        model,
        test_loader,
        device
    )

    results.append({
        "file_name": file_name,
        "accuracy": round(accuracy, 4),
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "recall" : round(recall, 4),
        "presicion" : round(precision, 4),
        "time" : round(mean_time,4),
        "parameters" : param
    })

df = pd.DataFrame(results)

df = df.sort_values(by="file_name", ascending=False)

print("\nRESULTS:\n")

print(df.to_string(index=False))

['resnet50_100.pth', 'resnet50_25.pth', 'resnet50_25_noise.pth', 'resnet50_50.pth', 'resnet50_50_noise.pth', 'resnet50_75_noise.pth']


C:\Users\kubac\AppData\Local\Temp\ipykernel_33380\3638666515.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)
C:\Users


RESULTS:

            file_name  accuracy  macro_f1  micro_f1  recall  presicion   time  parameters
resnet50_75_noise.pth    0.9958    0.9941    0.9958  0.9958     0.9958 0.0128    23585894
resnet50_50_noise.pth    0.9958    0.9943    0.9958  0.9958     0.9958 0.0129    23585894
      resnet50_50.pth    0.9945    0.9927    0.9945  0.9945     0.9945 0.0125    23585894
resnet50_25_noise.pth    0.9965    0.9941    0.9965  0.9965     0.9965 0.0124    23585894
      resnet50_25.pth    0.9819    0.9751    0.9819  0.9819     0.9819 0.0123    23585894
     resnet50_100.pth    0.9960    0.9941    0.9960  0.9960     0.9960 0.0124    23585894


In [56]:
import pandas as pd

import os

MODEL_DIR = fr"output"
MODEL_PREFIX = "mobilenetv2"

model_files = [
    f for f in os.listdir(MODEL_DIR)
    if f.endswith(".pth") and f.startswith(MODEL_PREFIX)
]
print(model_files)

results = []
for file_name in model_files:

    path = os.path.join(MODEL_DIR, file_name)


    model = models.mobilenet_v2(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 38)

    state_dict = torch.load(path, map_location=device)

    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(state_dict)

    model.to(device)
    model.eval()

    accuracy, macro_f1, micro_f1, recall, precision, mean_time, param = evaluate_model(
        model,
        test_loader,
        device
    )

    results.append({
        "file_name": file_name,
        "accuracy": round(accuracy, 4),
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "recall" : round(recall, 4),
        "presicion" : round(precision, 4),
        "time" : round(mean_time,4),
        "parameters" : param
    })

df = pd.DataFrame(results)

df = df.sort_values(by="file_name", ascending=False)

print("\nRESULTS:\n")

print(df.to_string(index=False))

['mobilenetv2_100.pth', 'mobilenetv2_25.pth', 'mobilenetv2_25_noise.pth', 'mobilenetv2_50.pth', 'mobilenetv2_50_noise.pth', 'mobilenetv2_75_noise.pth']


C:\Users\kubac\AppData\Local\Temp\ipykernel_33380\1845573988.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)
C:\Users


RESULTS:

               file_name  accuracy  macro_f1  micro_f1  recall  presicion   time  parameters
mobilenetv2_75_noise.pth    0.9965    0.9950    0.9965  0.9965     0.9965 0.0102     2272550
mobilenetv2_50_noise.pth    0.9951    0.9931    0.9951  0.9951     0.9951 0.0106     2272550
      mobilenetv2_50.pth    0.9943    0.9902    0.9943  0.9943     0.9943 0.0103     2272550
mobilenetv2_25_noise.pth    0.9978    0.9967    0.9978  0.9978     0.9978 0.0107     2272550
      mobilenetv2_25.pth    0.9903    0.9866    0.9903  0.9903     0.9903 0.0106     2272550
     mobilenetv2_100.pth    0.9951    0.9925    0.9951  0.9951     0.9951 0.0109     2272550


In [57]:
import pandas as pd

import os

MODEL_DIR = fr"output"
MODEL_PREFIX = "mobilenetv3"

model_files = [
    f for f in os.listdir(MODEL_DIR)
    if f.endswith(".pth") and f.startswith(MODEL_PREFIX)
]
print(model_files)

results = []
for file_name in model_files:

    path = os.path.join(MODEL_DIR, file_name)


    model = models.mobilenet_v3_small(weights=None)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, 38)

    state_dict = torch.load(path, map_location=device)

    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(state_dict)

    model.to(device)
    model.eval()

    accuracy, macro_f1, micro_f1, recall, precision, mean_time, param = evaluate_model(
        model,
        test_loader,
        device
    )

    results.append({
        "file_name": file_name,
        "accuracy": round(accuracy, 4),
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "recall" : round(recall, 4),
        "presicion" : round(precision, 4),
        "time" : round(mean_time,4),
        "parameters" : param
    })

df = pd.DataFrame(results)

df = df.sort_values(by="file_name", ascending=False)

print("\nRESULTS:\n")

print(df.to_string(index=False))

['mobilenetv3_100.pth', 'mobilenetv3_25.pth', 'mobilenetv3_25_noise.pth', 'mobilenetv3_50.pth', 'mobilenetv3_50_noise.pth', 'mobilenetv3_75_noise.pth']


C:\Users\kubac\AppData\Local\Temp\ipykernel_33380\1982144096.py:24: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)
C:\Users


RESULTS:

               file_name  accuracy  macro_f1  micro_f1  recall  presicion   time  parameters
mobilenetv3_75_noise.pth    0.9921    0.9870    0.9921  0.9921     0.9921 0.0102     1556806
mobilenetv3_50_noise.pth    0.9940    0.9910    0.9940  0.9940     0.9940 0.0103     1556806
      mobilenetv3_50.pth    0.9874    0.9801    0.9874  0.9874     0.9874 0.0102     1556806
mobilenetv3_25_noise.pth    0.9940    0.9905    0.9940  0.9940     0.9940 0.0103     1556806
      mobilenetv3_25.pth    0.9828    0.9775    0.9828  0.9828     0.9828 0.0105     1556806
     mobilenetv3_100.pth    0.9954    0.9921    0.9954  0.9954     0.9954 0.0106     1556806


In [58]:
import pandas as pd

import os

MODEL_DIR = fr"output"
MODEL_PREFIX = "efficientnetb0"

model_files = [
    f for f in os.listdir(MODEL_DIR)
    if f.endswith(".pth") and f.startswith(MODEL_PREFIX)
]
print(model_files)

results = []
for file_name in model_files:

    path = os.path.join(MODEL_DIR, file_name)


    model = models.efficientnet_b0(weights=None)

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 38)
    model = model.to(device)

    state_dict = torch.load(path, map_location=device)

    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(state_dict)

    model.to(device)
    model.eval()

    accuracy, macro_f1, micro_f1, recall, precision, mean_time, param = evaluate_model(
        model,
        test_loader,
        device
    )

    results.append({
        "file_name": file_name,
        "accuracy": round(accuracy, 4),
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "recall" : round(recall, 4),
        "presicion" : round(precision, 4),
        "time" : round(mean_time,4),
        "parameters" : param
    })

df = pd.DataFrame(results)

df = df.sort_values(by="file_name", ascending=False)

print("\nRESULTS:\n")

print(df.to_string(index=False))

['efficientnetb0_100.pth', 'efficientnetb0_25.pth', 'efficientnetb0_25_noise.pth', 'efficientnetb0_50.pth', 'efficientnetb0_50_noise.pth', 'efficientnetb0_75_noise.pth']


C:\Users\kubac\AppData\Local\Temp\ipykernel_33380\2733778520.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)
C:\Users


RESULTS:

                  file_name  accuracy  macro_f1  micro_f1  recall  presicion   time  parameters
efficientnetb0_75_noise.pth    0.9967    0.9953    0.9967  0.9967     0.9967 0.0161     4056226
efficientnetb0_50_noise.pth    0.9971    0.9956    0.9971  0.9971     0.9971 0.0167     4056226
      efficientnetb0_50.pth    0.9973    0.9951    0.9973  0.9973     0.9973 0.0163     4056226
efficientnetb0_25_noise.pth    0.9978    0.9962    0.9978  0.9978     0.9978 0.0165     4056226
      efficientnetb0_25.pth    0.9903    0.9855    0.9903  0.9903     0.9903 0.0156     4056226
     efficientnetb0_100.pth    0.9965    0.9937    0.9965  0.9965     0.9965 0.0156     4056226


In [59]:
test_val_transform = transforms.Compose([
        transforms.Resize((240,240)),
        transforms.ToTensor(),
])

test_ds  = datasets.ImageFolder(fr"michal/nowe/100/test", transform=test_val_transform)

test_loader  = DataLoader(test_ds, batch_size=BATCH)

import pandas as pd

import os

MODEL_DIR = fr"output"
MODEL_PREFIX = "efficientnetb1"

model_files = [
    f for f in os.listdir(MODEL_DIR)
    if f.endswith(".pth") and f.startswith(MODEL_PREFIX)
]
print(model_files)

results = []
for file_name in model_files:

    path = os.path.join(MODEL_DIR, file_name)


    model = models.efficientnet_b1(weights=None)

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 38)
    model = model.to(device)

    state_dict = torch.load(path, map_location=device)

    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(state_dict)
    model.eval()

    model.to(device)

    accuracy, macro_f1, micro_f1, recall, precision, mean_time, param = evaluate_model(
        model,
        test_loader,
        device
    )

    results.append({
        "file_name": file_name,
        "accuracy": round(accuracy, 4),
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "recall" : round(recall, 4),
        "presicion" : round(precision, 4),
        "time" : round(mean_time,4),
        "parameters" : param
    })

df = pd.DataFrame(results)

df = df.sort_values(by="file_name", ascending=False)

print("\nRESULTS:\n")

print(df.to_string(index=False))

['efficientnetb1_100.pth', 'efficientnetb1_25.pth', 'efficientnetb1_25_noise.pth', 'efficientnetb1_50.pth', 'efficientnetb1_50_noise.pth', 'efficientnetb1_75_noise.pth']


C:\Users\kubac\AppData\Local\Temp\ipykernel_33380\2137483450.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)
C:\Users


RESULTS:

                  file_name  accuracy  macro_f1  micro_f1  recall  presicion   time  parameters
efficientnetb1_75_noise.pth    0.9969    0.9947    0.9969  0.9969     0.9969 0.0234     6561862
efficientnetb1_50_noise.pth    0.9958    0.9945    0.9958  0.9958     0.9958 0.0232     6561862
      efficientnetb1_50.pth    0.9932    0.9893    0.9932  0.9932     0.9932 0.0232     6561862
efficientnetb1_25_noise.pth    0.9962    0.9942    0.9962  0.9962     0.9962 0.0229     6561862
      efficientnetb1_25.pth    0.9910    0.9869    0.9910  0.9910     0.9910 0.0229     6561862
     efficientnetb1_100.pth    0.9973    0.9946    0.9973  0.9973     0.9973 0.0226     6561862


In [60]:
test_val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE,IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]            
        )
    ])

test_ds  = datasets.ImageFolder(fr"michal/nowe/100/test", transform=test_val_transform)

test_loader  = DataLoader(test_ds, batch_size=BATCH)

MODEL_DIR = fr"output"
MODEL_PREFIX = "vit_b_16"

model_files = [
    f for f in os.listdir(MODEL_DIR)
    if f.endswith(".pth") and f.startswith(MODEL_PREFIX)
]
print(model_files)

results = []
for file_name in model_files:

    path = os.path.join(MODEL_DIR, file_name)


    model = models.vit_b_16(weights=None)

    model.heads.head = nn.Linear(model.heads.head.in_features, 38)
    model = model.to(device)

    state_dict = torch.load(path, map_location=device)

    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(state_dict)
    model.eval()

    model.to(device)

    accuracy, macro_f1, micro_f1, recall, precision, mean_time, param = evaluate_model(
        model,
        test_loader,
        device
    )

    results.append({
        "file_name": file_name,
        "accuracy": round(accuracy, 4),
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "recall" : round(recall, 4),
        "presicion" : round(precision, 4),
        "time" : round(mean_time,4),
        "parameters" : param
    })

df = pd.DataFrame(results)

df = df.sort_values(by="file_name", ascending=False)

print("\nRESULTS:\n")

print(df.to_string(index=False))

['vit_b_16_100.pth', 'vit_b_16_25.pth', 'vit_b_16_25_noise.pth', 'vit_b_16_50.pth', 'vit_b_16_50_noise.pth', 'vit_b_16_75_noise.pth']


C:\Users\kubac\AppData\Local\Temp\ipykernel_33380\660681362.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)
C:\Users\


RESULTS:

            file_name  accuracy  macro_f1  micro_f1  recall  presicion   time  parameters
vit_b_16_75_noise.pth    0.9938    0.9890    0.9938  0.9938     0.9938 0.0102    85827878
vit_b_16_50_noise.pth    0.9967    0.9961    0.9967  0.9967     0.9967 0.0102    85827878
      vit_b_16_50.pth    0.9925    0.9867    0.9925  0.9925     0.9925 0.0099    85827878
vit_b_16_25_noise.pth    0.9963    0.9941    0.9963  0.9963     0.9963 0.0101    85827878
      vit_b_16_25.pth    0.9857    0.9803    0.9857  0.9857     0.9857 0.0099    85827878
     vit_b_16_100.pth    0.9958    0.9928    0.9958  0.9958     0.9958 0.0097    85827878


In [61]:
test_val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE,IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]            
        )
    ])


test_ds  = datasets.ImageFolder(fr"michal/nowe/100/test", transform=test_val_transform)

test_loader  = DataLoader(test_ds, batch_size=BATCH)

MODEL_DIR = fr"output"
MODEL_PREFIX = "vit_b_32"

model_files = [
    f for f in os.listdir(MODEL_DIR)
    if f.endswith(".pth") and f.startswith(MODEL_PREFIX)
]
print(model_files)

results = []
for file_name in model_files:

    path = os.path.join(MODEL_DIR, file_name)

    model = models.vit_b_32(weights=None)

    model.heads.head = nn.Linear(model.heads.head.in_features, 38)
    model = model.to(device)

    state_dict = torch.load(path, map_location=device)

    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(state_dict)
    model.eval()

    model.to(device)

    accuracy, macro_f1, micro_f1, recall, precision, mean_time, param = evaluate_model(
        model,
        test_loader,
        device
    )

    results.append({
        "file_name": file_name,
        "accuracy": round(accuracy, 4),
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "recall" : round(recall, 4),
        "presicion" : round(precision, 4),
        "time" : round(mean_time,4),
        "parameters" : param
    })

df = pd.DataFrame(results)

df = df.sort_values(by="file_name", ascending=False)

print("\nRESULTS:\n")

print(df.to_string(index=False))

['vit_b_32_100.pth', 'vit_b_32_25.pth', 'vit_b_32_25_noise.pth', 'vit_b_32_50.pth', 'vit_b_32_50_noise.pth', 'vit_b_32_75_noise.pth']


C:\Users\kubac\AppData\Local\Temp\ipykernel_33380\1472246849.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)
C:\Users


RESULTS:

            file_name  accuracy  macro_f1  micro_f1  recall  presicion   time  parameters
vit_b_32_75_noise.pth    0.9890    0.9840    0.9890  0.9890     0.9890 0.0104    87484454
vit_b_32_50_noise.pth    0.9934    0.9905    0.9934  0.9934     0.9934 0.0103    87484454
      vit_b_32_50.pth    0.9883    0.9859    0.9883  0.9883     0.9883 0.0103    87484454
vit_b_32_25_noise.pth    0.9868    0.9837    0.9868  0.9868     0.9868 0.0104    87484454
      vit_b_32_25.pth    0.9786    0.9705    0.9786  0.9786     0.9786 0.0110    87484454
     vit_b_32_100.pth    0.9875    0.9846    0.9875  0.9875     0.9875 0.0108    87484454


In [62]:
test_val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE,IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]            
        )
    ])


test_ds  = datasets.ImageFolder(fr"michal/nowe/100/test", transform=test_val_transform)

test_loader  = DataLoader(test_ds, batch_size=BATCH)

MODEL_DIR = fr"output"
MODEL_PREFIX = "DenseNet121"

model_files = [
    f for f in os.listdir(MODEL_DIR)
    if f.endswith(".pth") and f.startswith(MODEL_PREFIX)
]
print(model_files)

results = []
for file_name in model_files:

    path = os.path.join(MODEL_DIR, file_name)

    model = models.densenet121(weights=None)

    model.classifier = nn.Linear(model.classifier.in_features,38)
    model = model.to(device)

    state_dict = torch.load(path, map_location=device)

    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(state_dict)
    model.eval()

    model.to(device)

    accuracy, macro_f1, micro_f1, recall, precision, mean_time, param = evaluate_model(
        model,
        test_loader,
        device
    )

    results.append({
        "file_name": file_name,
        "accuracy": round(accuracy, 4),
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "recall" : round(recall, 4),
        "presicion" : round(precision, 4),
        "time" : round(mean_time,4),
        "parameters" : param
    })

df = pd.DataFrame(results)

df = df.sort_values(by="file_name", ascending=False)

print("\nRESULTS:\n")

print(df.to_string(index=False))

['DenseNet121_100.pth', 'DenseNet121_25.pth', 'DenseNet121_50.pth']


C:\Users\kubac\AppData\Local\Temp\ipykernel_33380\3433988146.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)
C:\Users


RESULTS:

          file_name  accuracy  macro_f1  micro_f1  recall  presicion   time  parameters
 DenseNet121_50.pth    0.6807    0.6235    0.6807  0.6807     0.6807 0.0317     6992806
 DenseNet121_25.pth    0.8102    0.7676    0.8102  0.8102     0.8102 0.0315     6992806
DenseNet121_100.pth    0.4917    0.4530    0.4917  0.4917     0.4917 0.0321     6992806


In [63]:
test_val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE,IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]            
        )
    ])


test_ds  = datasets.ImageFolder(fr"michal/nowe/100/test", transform=test_val_transform)

test_loader  = DataLoader(test_ds, batch_size=BATCH)

MODEL_DIR = fr"output"
MODEL_PREFIX = "densenet169"

model_files = [
    f for f in os.listdir(MODEL_DIR)
    if f.endswith(".pth") and f.startswith(MODEL_PREFIX)
]
print(model_files)

results = []
for file_name in model_files:

    path = os.path.join(MODEL_DIR, file_name)

    model = models.densenet169(weights=None)

    model.classifier = nn.Linear(model.classifier.in_features,38)
    model = model.to(device)

    state_dict = torch.load(path, map_location=device)

    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(state_dict)
    model.eval()

    model.to(device)

    accuracy, macro_f1, micro_f1, recall, precision, mean_time, param = evaluate_model(
        model,
        test_loader,
        device
    )

    results.append({
        "file_name": file_name,
        "accuracy": round(accuracy, 4),
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "recall" : round(recall, 4),
        "presicion" : round(precision, 4),
        "time" : round(mean_time,4),
        "parameters" : param
    })

df = pd.DataFrame(results)

df = df.sort_values(by="file_name", ascending=False)

print("\nRESULTS:\n")

print(df.to_string(index=False))

['densenet169_25_noise.pth', 'densenet169_50_noise.pth', 'densenet169_75_noise.pth']


C:\Users\kubac\AppData\Local\Temp\ipykernel_33380\3291596868.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)
C:\Users


RESULTS:

               file_name  accuracy  macro_f1  micro_f1  recall  presicion   time  parameters
densenet169_75_noise.pth    0.6433    0.5799    0.6433  0.6433     0.6433 0.0450    12547750
densenet169_50_noise.pth    0.6375    0.5717    0.6375  0.6375     0.6375 0.0449    12547750
densenet169_25_noise.pth    0.6356    0.5581    0.6356  0.6356     0.6356 0.0449    12547750


In [64]:
test_val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE,IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]            
        )
    ])


test_ds  = datasets.ImageFolder(fr"michal/nowe/100/test", transform=test_val_transform)

test_loader  = DataLoader(test_ds, batch_size=BATCH)

MODEL_DIR = fr"output"
MODEL_PREFIX = "DenseNet169"

model_files = [
    f for f in os.listdir(MODEL_DIR)
    if f.endswith(".pth") and f.startswith(MODEL_PREFIX)
]
print(model_files)

results = []
for file_name in model_files:

    path = os.path.join(MODEL_DIR, file_name)

    model = models.densenet169(weights=None)

    model.classifier = nn.Linear(model.classifier.in_features,38)
    model = model.to(device)

    state_dict = torch.load(path, map_location=device)

    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(state_dict)
    model.eval()

    model.to(device)

    accuracy, macro_f1, micro_f1, recall, precision, mean_time, param = evaluate_model(
        model,
        test_loader,
        device
    )

    results.append({
        "file_name": file_name,
        "accuracy": round(accuracy, 4),
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "recall" : round(recall, 4),
        "presicion" : round(precision, 4),
        "time" : round(mean_time,4),
        "parameters" : param
    })

df = pd.DataFrame(results)

df = df.sort_values(by="file_name", ascending=False)

print("\nRESULTS:\n")

print(df.to_string(index=False))

['DenseNet169_100.pth', 'DenseNet169_25.pth', 'DenseNet169_50.pth']


C:\Users\kubac\AppData\Local\Temp\ipykernel_33380\3951492447.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)
C:\Users


RESULTS:

          file_name  accuracy  macro_f1  micro_f1  recall  presicion   time  parameters
 DenseNet169_50.pth    0.7327    0.7032    0.7327  0.7327     0.7327 0.0449    12547750
 DenseNet169_25.pth    0.8676    0.8275    0.8676  0.8676     0.8676 0.0450    12547750
DenseNet169_100.pth    0.7560    0.7042    0.7560  0.7560     0.7560 0.0451    12547750


In [65]:
test_val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE,IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]            
        )
    ])


test_ds  = datasets.ImageFolder(fr"michal/nowe/100/test", transform=test_val_transform)

test_loader  = DataLoader(test_ds, batch_size=BATCH)

MODEL_DIR = fr"output"
MODEL_PREFIX = "densenet121"

model_files = [
    f for f in os.listdir(MODEL_DIR)
    if f.endswith(".pth") and f.startswith(MODEL_PREFIX)
]
print(model_files)

results = []
for file_name in model_files:

    path = os.path.join(MODEL_DIR, file_name)

    model = models.densenet121(weights=None)

    model.classifier = nn.Linear(model.classifier.in_features,38)
    model = model.to(device)

    state_dict = torch.load(path, map_location=device)

    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]

    model.load_state_dict(state_dict)
    model.eval()

    model.to(device)

    accuracy, macro_f1, micro_f1, recall, precision, mean_time, param = evaluate_model(
        model,
        test_loader,
        device
    )

    results.append({
        "file_name": file_name,
        "accuracy": round(accuracy, 4),
        "macro_f1": round(macro_f1, 4),
        "micro_f1": round(micro_f1, 4),
        "recall" : round(recall, 4),
        "presicion" : round(precision, 4),
        "time" : round(mean_time,4),
        "parameters" : param
    })

df = pd.DataFrame(results)

df = df.sort_values(by="file_name", ascending=False)

print("\nRESULTS:\n")

print(df.to_string(index=False))

['densenet121_25_noise.pth', 'densenet121_50_noise.pth', 'densenet121_75_noise.pth']


C:\Users\kubac\AppData\Local\Temp\ipykernel_33380\2672500709.py:34: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path, map_location=device)
C:\Users


RESULTS:

               file_name  accuracy  macro_f1  micro_f1  recall  presicion   time  parameters
densenet121_75_noise.pth    0.5268    0.4836    0.5268  0.5268     0.5268 0.0317     6992806
densenet121_50_noise.pth    0.5701    0.5013    0.5701  0.5701     0.5701 0.0318     6992806
densenet121_25_noise.pth    0.5464    0.5265    0.5464  0.5464     0.5464 0.0318     6992806
